Part 1: Data Generation

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Set seed for reproducibility
np.random.seed(42)
num_rows = 600  # Ensuring at least 500+ rows

# 1. Customers Dataset
customer_ids = [f"CUST_{str(i).zfill(4)}" for i in range(1, 151)]
customer_types = ['REGULAR', 'PREMIUM', 'VIP']
emails = [f"user_{i}@example.com" for i in range(1, 151)]

# Inject 2% invalid emails
for idx in np.random.choice(len(emails), size=int(0.02 * len(emails)), replace=False):
    emails[idx] = emails[idx].replace("@", "")  # Missing @

customers_df = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_name': [f"Customer {i}" for i in range(1, 151)],
    'email': emails,
    'registration_date': [pd.Timestamp('2024-01-01') + pd.Timedelta(days=int(np.random.randint(0, 365))) for _ in range(150)],
    'customer_type': np.random.choice(customer_types, size=150)
})

# 2. Products Dataset
categories = {
    'Electronics': ['Laptop', 'Smartphone', 'Headphones', 'Smartwatch'],
    'Clothing': ['T-Shirt', 'Jeans', 'Jacket', 'Sneakers'],
    'Home': ['Lamp', 'Blender', 'Coffee Maker', 'Rug'],
    'Books': ['Fiction Novel', 'Sci-Fi Book', 'Biography', 'Textbook']
}

product_records = []
p_id = 1
for cat, subcats in categories.items():
    for subcat in subcats:
        # Intentionally creating mixed case and extra spaces
        p_name = f"  {cat[:3].upper()}_{subcat.lower()}  " 
        product_records.append({
            'product_id': f"PROD_{str(p_id).zfill(4)}",
            'product_name': p_name,
            'category': cat,
            'subcategory': subcat,
            'cost_price': round(np.random.uniform(10, 500), 2)
        })
        p_id += 1
products_df = pd.DataFrame(product_records)

# 3. Orders Dataset
order_ids = [f"ORD_{str(i).zfill(4)}" for i in range(1, num_rows + 1)]
regions = ['US', 'EU', 'APAC', 'LATAM']
statuses = ['PLACED', 'SHIPPED', 'DELIVERED', 'CANCELLED', 'RETURNED']

order_dates = []
for _ in range(num_rows):
    dt = pd.Timestamp('2024-06-01') + pd.Timedelta(days=int(np.random.randint(0, 400)), hours=int(np.random.randint(0, 24)))
    # Inject wrong date format (DD-MM-YYYY) for ~5% of rows
    if np.random.rand() < 0.05:
        order_dates.append(dt.strftime('%d-%m-%Y'))
    else:
        order_dates.append(dt.strftime('%Y-%m-%Y %H:%M:%S'))

cust_ids_pool = customer_ids.copy()
# Inject 5% NULL customer IDs
cust_ids_with_nulls = [np.random.choice(cust_ids_pool) if np.random.rand() > 0.05 else None for _ in range(num_rows)]

orders_df = pd.DataFrame({
    'order_id': order_ids,
    'customer_id': cust_ids_with_nulls,
    'order_date': order_dates,
    'status': np.random.choice(statuses, size=num_rows, p=[0.2, 0.3, 0.3, 0.1, 0.1]),
    'region_code': np.random.choice(regions, size=num_rows)
})

# 4. Order Items Dataset
item_records = []
item_id_counter = 1
prod_ids = products_df['product_id'].tolist()

for o_id in order_ids:
    # Ensure referential integrity for most, but we will inject orphaned records later
    num_items = np.random.randint(1, 4)
    for _ in range(num_items):
        qty = np.random.randint(1, 5)
        # 3% negative quantities (returns)
        if np.random.rand() < 0.03:
            qty = -qty
        
        item_records.append({
            'item_id': f"ITEM_{str(item_id_counter).zfill(5)}",
            'order_id': o_id,
            'product_id': np.random.choice(prod_ids),
            'quantity': qty,
            'unit_price': round(np.random.uniform(15, 600), 2),
            'discount_percent': round(np.random.uniform(0, 30), 2)
        })
        item_id_counter += 1

# Inject referential integrity error (orphaned order items)
for _ in range(5):
    item_records.append({
        'item_id': f"ITEM_{str(item_id_counter).zfill(5)}",
        'order_id': "ORD_9999",  # Non-existent order ID
        'product_id': np.random.choice(prod_ids),
        'quantity': np.random.randint(1, 3),
        'unit_price': 50.00,
        'discount_percent': 0.0
    })
    item_id_counter += 1

order_items_df = pd.DataFrame(item_records)

# Save to Databricks Driver Storage locally first
os.makedirs('/tmp/ecommerce_data/', exist_ok=True)
orders_df.to_csv('/tmp/ecommerce_data/orders.csv', index=False)
order_items_df.to_csv('/tmp/ecommerce_data/order_items.csv', index=False)
products_df.to_csv('/tmp/ecommerce_data/products.csv', index=False)
customers_df.to_csv('/tmp/ecommerce_data/customers.csv', index=False)

print("Data Generation Complete! Files saved to /tmp/ecommerce_data/")

Data Generation Complete! Files saved to /tmp/ecommerce_data/


Part 2: Data Cleaning & Validation

In [0]:
import re

issues_log = []

def clean_orders(df):a
    cleaned_df = df.copy()
    
    # Track missing customer IDs
    missing_cust = cleaned_df[cleaned_df['customer_id'].isna() | (cleaned_df['customer_id'] == 'NULL')]
    issues_log.append(f"Orders: Found {len(missing_cust)} rows with missing customer_id. Dropping rows for referential safety.")
    cleaned_df = cleaned_df.dropna(subset=['customer_id'])
    
    # Fix date formats
    def parse_date(date_str):
        try:
            # Try parsing normal format
            return pd.to_datetime(date_str, format='%Y-%m-%d %H:%M:%S')
        except:
            try:
                # Try parsing the messy DD-MM-YYYY format
                return pd.to_datetime(date_str, format='%d-%m-%Y')
            except:
                return pd.NaT

    cleaned_df['order_date'] = cleaned_df['order_date'].apply(parse_date)
    bad_dates = cleaned_df['order_date'].isna().sum()
    if bad_dates > 0:
        issues_log.append(f"Orders: Found {bad_dates} unparseable dates.")
    
    return cleaned_df.dropna(subset=['order_date'])

def clean_products(df):
    cleaned_df = df.copy()
    # Normalize spaces and change case to Title Case
    cleaned_df['product_name'] = cleaned_df['product_name'].str.strip().str.title()
    issues_log.append("Products: Processed text cleaning (Trimmed spaces & applied Title Case).")
    return cleaned_df

def validate_emails(df):
    invalid_ids = []
    email_regex = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    
    for idx, row in df.iterrows():
        if not re.match(email_regex, str(row['email'])):
            invalid_ids.append(row['customer_id'])
            
    issues_log.append(f"Customers: Identified {len(invalid_ids)} customers with invalid emails: {invalid_ids}")
    return invalid_ids

def check_referential_integrity(items_df, base_orders_df):
    valid_order_ids = set(base_orders_df['order_id'])
    orphaned_items = items_df[~items_df['order_id'].isin(valid_order_ids)]
    
    if len(orphaned_items) > 0:
        issues_log.append(f"Order Items: Found {len(orphaned_items)} records referencing non-existent order IDs.")
        
    # Keep only records that map perfectly to the orders table
    cleaned_items_df = items_df[items_df['order_id'].isin(valid_order_ids)]
    return cleaned_items_df

# Run execution workflow
raw_orders = pd.read_csv('/tmp/ecommerce_data/orders.csv')
raw_items = pd.read_csv('/tmp/ecommerce_data/order_items.csv')
raw_products = pd.read_csv('/tmp/ecommerce_data/products.csv')
raw_customers = pd.read_csv('/tmp/ecommerce_data/customers.csv')

# Execute pipelines
cleaned_orders_df = clean_orders(raw_orders)
cleaned_products_df = clean_products(raw_products)
invalid_customer_emails = validate_emails(raw_customers)
cleaned_items_df = check_referential_integrity(raw_items, cleaned_orders_df)

# Print out issues report
print("\n--- ANOMALY PIPELINE REPORT ---")
for log in issues_log:
    print(f"[ISSUE IDENTIFIED] {log}")

# Write to Delta Tables inside Databricks Environment
spark.createDataFrame(cleaned_orders_df).write.format("delta").mode("overwrite").saveAsTable("orders")
spark.createDataFrame(cleaned_items_df).write.format("delta").mode("overwrite").saveAsTable("order_items")
spark.createDataFrame(cleaned_products_df).write.format("delta").mode("overwrite").saveAsTable("products")
spark.createDataFrame(raw_customers).write.format("delta").mode("overwrite").saveAsTable("customers")

print("\nAll clean datasets have been registered as managed Delta Lake Tables!")


--- ANOMALY PIPELINE REPORT ---
[ISSUE IDENTIFIED] Orders: Found 32 rows with missing customer_id. Dropping rows for referential safety.
[ISSUE IDENTIFIED] Orders: Found 541 unparseable dates.
[ISSUE IDENTIFIED] Products: Processed text cleaning (Trimmed spaces & applied Title Case).
[ISSUE IDENTIFIED] Customers: Identified 3 customers with invalid emails: ['CUST_0019', 'CUST_0074', 'CUST_0119']
[ISSUE IDENTIFIED] Order Items: Found 1142 records referencing non-existent order IDs.

All clean datasets have been registered as managed Delta Lake Tables!


Part 3: Spark SQL Analysis

1. Total revenue per category

In [0]:
%sql
SELECT 
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

category,total_revenue
Books,10175.66
Electronics,7137.41
Home,6402.78
Clothing,4862.67


2. Top 10 customers by total order value

In [0]:
%sql
SELECT 
    o.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) AS total_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;

customer_id,customer_name,total_order_value
CUST_0093,Customer 93,3622.32
CUST_0131,Customer 131,2903.62
CUST_0063,Customer 63,2829.34
CUST_0024,Customer 24,2411.02
CUST_0143,Customer 143,2051.44
CUST_0062,Customer 62,1851.17
CUST_0103,Customer 103,1759.14
CUST_0105,Customer 105,1656.59
CUST_0119,Customer 119,1137.91
CUST_0127,Customer 127,1030.19


3. Month-wise order count for the last 12 months

In [0]:
%sql
SELECT 
    DATE_FORMAT(order_date, 'yyyy-MM') AS order_month,
    COUNT(DISTINCT order_id) AS total_orders
FROM orders
WHERE order_date >= ADD_MONTHS(CURRENT_DATE(), -12)
GROUP BY order_month
ORDER BY order_month;

order_month,total_orders


Queries
4. Find customers who placed orders but never had any item delivered

In [0]:
%sql
SELECT DISTINCT o.customer_id, c.customer_name 
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.customer_id NOT IN (
    SELECT DISTINCT customer_id 
    FROM orders 
    WHERE status = 'DELIVERED'
);

customer_id,customer_name
CUST_0105,Customer 105
CUST_0063,Customer 63
CUST_0049,Customer 49
CUST_0103,Customer 103
CUST_0124,Customer 124
CUST_0129,Customer 129
CUST_0093,Customer 93
CUST_0119,Customer 119
CUST_0127,Customer 127
CUST_0062,Customer 62


5. Products that were ordered but had more returns than purchases

In [0]:
%sql
SELECT 
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS total_purchased,
    SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS total_returned
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING total_returned > total_purchased;

product_id,product_name,total_purchased,total_returned


6. Calculate the return rate per category

In [0]:
%sql
SELECT 
    p.category,
    SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returned_items,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased_items,
    ROUND(SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) / 
          NULLIF(SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END), 0), 4) AS return_rate
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate DESC;

category,returned_items,purchased_items,return_rate
Home,3,24,0.125
Books,3,37,0.0811
Electronics,0,31,0.0
Clothing,0,26,0.0


7. Running Totals with Window Functions

In [0]:
%sql
WITH DailyRevenue AS (
    SELECT 
        o.region_code,
        TO_DATE(o.order_date) AS order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.region_code, TO_DATE(o.order_date)
)
SELECT 
    region_code,
    order_date,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(SUM(daily_revenue) OVER(PARTITION BY region_code ORDER BY order_date), 2) AS running_total
FROM DailyRevenue
ORDER BY region_code, order_date;

region_code,order_date,daily_revenue,running_total
APAC,2024-11-20,981.56,981.56
APAC,2025-06-12,954.11,1935.67
EU,2024-07-02,536.78,536.78
EU,2024-08-26,571.02,1107.8
EU,2024-09-01,2102.0,3209.8
EU,2025-01-12,338.12,3547.92
EU,2025-05-06,690.66,4238.58
EU,2025-05-30,1656.59,5895.17
LATAM,2024-09-16,372.37,372.37
LATAM,2024-11-09,187.31,559.69


8. Ranking with DENSE_RANK

In [0]:
%sql
WITH ProductRev AS (
    SELECT 
        p.category,
        p.product_name,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
SELECT 
    category,
    product_name,
    total_revenue,
    DENSE_RANK() OVER(PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM ProductRev;

category,product_name,total_revenue,rank_in_category
Books,Boo_Fiction Novel,3716.14,1
Books,Boo_Biography,3418.17,2
Books,Boo_Sci-Fi Book,1920.67,3
Books,Boo_Textbook,1120.67,4
Clothing,Clo_Sneakers,1767.29,1
Clothing,Clo_Jacket,1252.68,2
Clothing,Clo_Jeans,1107.03,3
Clothing,Clo_T-Shirt,735.66,4
Electronics,Ele_Headphones,2035.87,1
Electronics,Ele_Laptop,1965.79,2


# 9. LAG/LEAD Analysis

In [0]:
%sql
WITH OrderedTimeline AS (
    SELECT 
        customer_id,
        TO_DATE(order_date) AS order_date,
        LAG(TO_DATE(order_date)) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
),
GapsCalculated AS (
    SELECT 
        customer_id,
        order_date,
        previous_order_date,
        DATEDIFF(order_date, previous_order_date) AS days_gap
    FROM OrderedTimeline
)
SELECT 
    customer_id,
    AVG(days_gap) AS avg_days_gap,
    CASE WHEN AVG(days_gap) > 30 THEN 'At Risk' ELSE 'Active' END AS customer_status
FROM GapsCalculated
GROUP BY customer_id;

customer_id,avg_days_gap,customer_status
CUST_0024,null,Active
CUST_0047,8.0,Active
CUST_0049,null,Active
CUST_0059,null,Active
CUST_0062,null,Active
CUST_0063,null,Active
CUST_0076,null,Active
CUST_0079,null,Active
CUST_0080,null,Active
CUST_0084,null,Active


10. CTE with Multiple Levels

In [0]:
%sql
WITH CustomerMonthlyRevenue AS (
    SELECT 
        customer_id,
        DATE_FORMAT(order_date, 'yyyy-MM') AS order_month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)) AS monthly_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY customer_id, DATE_FORMAT(order_date, 'yyyy-MM')
),
CustomerSegments AS (
    SELECT 
        customer_id,
        order_month,
        CASE 
            WHEN monthly_revenue > 10000 THEN 'High'
            WHEN monthly_revenue BETWEEN 5000 AND 10000 THEN 'Medium'
            ELSE 'Low' 
        END AS segment
    FROM CustomerMonthlyRevenue
)
SELECT 
    order_month,
    segment,
    COUNT(customer_id) AS customer_count
FROM CustomerSegments
GROUP BY order_month, segment
ORDER BY order_month, segment;

order_month,segment,customer_count
2024-07,Low,1
2024-08,Low,1
2024-09,Low,2
2024-10,Low,1
2024-11,Low,4
2024-12,Low,2
2025-01,Low,1
2025-02,Low,1
2025-03,Low,4
2025-04,Low,2


11. NTILE for Segmentation

In [0]:
%sql
WITH CustomerLTV AS (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)) AS total_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.customer_id
),
Quartiles AS (
    SELECT 
        customer_id,
        total_value,
        NTILE(4) OVER (ORDER BY total_value DESC) AS quartile
    FROM CustomerLTV
)
SELECT 
    customer_id,
    ROUND(total_value, 2) AS total_value,
    quartile,
    CASE 
        WHEN quartile = 1 THEN 'Platinum'
        WHEN quartile = 2 THEN 'Gold'
        WHEN quartile = 3 THEN 'Silver'
        ELSE 'Bronze' 
    END AS quartile_label
FROM Quartiles;

customer_id,total_value,quartile,quartile_label
CUST_0093,3622.32,1,Platinum
CUST_0131,2903.62,1,Platinum
CUST_0063,2829.34,1,Platinum
CUST_0024,2411.02,1,Platinum
CUST_0143,2051.44,1,Platinum
CUST_0062,1851.17,1,Platinum
CUST_0103,1759.14,2,Gold
CUST_0105,1656.59,2,Gold
CUST_0119,1137.91,2,Gold
CUST_0127,1030.19,2,Gold


12. Year-over-Year Comparison

In [0]:
%sql
WITH MonthlyMetrics AS (
    SELECT 
        YEAR(o.order_date) AS yr,
        MONTH(o.order_date) AS mth,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY YEAR(o.order_date), MONTH(o.order_date)
)
SELECT 
    curr.yr AS year,
    curr.mth AS month,
    ROUND(curr.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    ROUND(((curr.revenue - COALESCE(prev.revenue, 0)) / NULLIF(prev.revenue, 0)) * 100, 2) AS yoy_growth_percent
FROM MonthlyMetrics curr
LEFT JOIN MonthlyMetrics prev ON curr.yr = prev.yr + 1 AND curr.mth = prev.mth
ORDER BY year, month;

year,month,revenue,prev_year_revenue,yoy_growth_percent
2024,7,536.78,null,null
2024,8,571.02,null,null
2024,9,2474.37,null,null
2024,10,649.96,null,null
2024,11,3185.85,null,null
2024,12,1447.19,null,null
2025,1,338.12,null,null
2025,2,1759.14,null,null
2025,3,9424.14,null,null
2025,4,975.18,null,null


13. First/Last Value Analysis

In [0]:
%sql
WITH OrderedPurchases AS (
    SELECT 
        o.customer_id,
        p.category,
        ROW_NUMBER() OVER(PARTITION BY o.customer_id ORDER BY o.order_date ASC) as rn_first,
        ROW_NUMBER() OVER(PARTITION BY o.customer_id ORDER BY o.order_date DESC) as rn_last
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
),
FirstCat AS (SELECT customer_id, category AS first_category FROM OrderedPurchases WHERE rn_first = 1),
LastCat AS (SELECT customer_id, category AS last_category FROM OrderedPurchases WHERE rn_last = 1)
SELECT 
    f.customer_id,
    f.first_category,
    l.last_category,
    CASE WHEN f.first_category = l.last_category THEN 'No' ELSE 'Yes' END AS category_shift
FROM FirstCat f
JOIN LastCat l ON f.customer_id = l.customer_id;

customer_id,first_category,last_category,category_shift
CUST_0024,Books,Books,No
CUST_0047,Electronics,Home,Yes
CUST_0049,Books,Books,No
CUST_0059,Clothing,Clothing,No
CUST_0062,Electronics,Electronics,No
CUST_0063,Electronics,Electronics,No
CUST_0076,Electronics,Electronics,No
CUST_0079,Home,Home,No
CUST_0080,Home,Home,No
CUST_0084,Clothing,Clothing,No


14. Cumulative Distribution

In [0]:
%sql
WITH CustomerRevenue AS (
    SELECT 
        customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY customer_id
),
TotalRunningRevenue AS (
    SELECT 
        customer_id,
        revenue,
        SUM(revenue) OVER(ORDER BY revenue DESC) AS cum_revenue,
        SUM(revenue) OVER() AS total_rev
    FROM CustomerRevenue
)
SELECT 
    customer_id,
    ROUND(revenue, 2) as revenue,
    ROUND(cum_revenue, 2) as cumulative_revenue,
    ROUND((cum_revenue / total_rev) * 100, 2) AS cumulative_percent
FROM TotalRunningRevenue
ORDER BY revenue DESC;

customer_id,revenue,cumulative_revenue,cumulative_percent
CUST_0093,3622.32,3622.32,12.67
CUST_0131,2903.62,6525.94,22.84
CUST_0063,2829.34,9355.28,32.74
CUST_0024,2411.02,11766.3,41.17
CUST_0143,2051.44,13817.73,48.35
CUST_0062,1851.17,15668.91,54.83
CUST_0103,1759.14,17428.05,60.98
CUST_0105,1656.59,19084.64,66.78
CUST_0119,1137.91,20222.55,70.76
CUST_0127,1030.19,21252.74,74.37


15. Complex CTE: Cohort Analysis

In [0]:
%sql
WITH Cohorts AS (
    SELECT 
        customer_id,
        DATE_FORMAT(registration_date, 'yyyy-MM') AS cohort_month
    FROM customers
),
Activity AS (
    SELECT 
        o.customer_id,
        DATE_FORMAT(o.order_date, 'yyyy-MM') AS order_month,
        c.cohort_month,
        (INT(SUBSTR(DATE_FORMAT(o.order_date, 'yyyy-MM'), 1, 4)) - INT(SUBSTR(c.cohort_month, 1, 4))) * 12 +
        (INT(SUBSTR(DATE_FORMAT(o.order_date, 'yyyy-MM'), 6, 2)) - INT(SUBSTR(c.cohort_month, 6, 2))) AS month_number
    FROM orders o
    JOIN Cohorts c ON o.customer_id = c.customer_id
),
CohortSizes AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_size FROM Cohorts GROUP BY cohort_month
)
SELECT 
    a.cohort_month,
    cs.cohort_size,
    COUNT(DISTINCT CASE WHEN a.month_number = 0 THEN a.customer_id END) AS month_0_users,
    COUNT(DISTINCT CASE WHEN a.month_number = 1 THEN a.customer_id END) AS month_1_users,
    COUNT(DISTINCT CASE WHEN a.month_number = 2 THEN a.customer_id END) AS month_2_users,
    COUNT(DISTINCT CASE WHEN a.month_number = 3 THEN a.customer_id END) AS month_3_users
FROM Activity a
JOIN CohortSizes cs ON a.cohort_month = cs.cohort_month
GROUP BY a.cohort_month, cs.cohort_size
ORDER BY a.cohort_month;

cohort_month,cohort_size,month_0_users,month_1_users,month_2_users,month_3_users
2024-01,6,0,0,0,0
2024-02,10,0,0,0,0
2024-03,5,0,0,0,0
2024-04,20,0,0,0,0
2024-05,22,0,0,0,1
2024-06,13,0,0,0,1
2024-07,13,0,0,0,0
2024-08,16,0,0,0,0
2024-09,11,0,0,0,0
2024-10,10,0,1,0,0


16. Self-Join with Window Function

In [0]:
%sql
WITH ItemPairs AS (
    SELECT 
        a.product_id AS product_a, 
        b.product_id AS product_b,
        a.order_id
    FROM order_items a
    JOIN order_items b ON a.order_id = b.order_id
    WHERE a.product_id < b.product_id  -- Prevents self-pairing and duplicates (A-B and B-A matching)
)
SELECT 
    pa.product_name AS product_a, 
    pb.product_name AS product_b, 
    COUNT(*) AS times_bought_together
FROM ItemPairs ip
JOIN products pa ON ip.product_a = pa.product_id
JOIN products pb ON ip.product_b = pb.product_id
GROUP BY pa.product_name, pb.product_name
ORDER BY times_bought_together DESC;

product_a,product_b,times_bought_together
Clo_Jeans,Clo_Sneakers,2
Ele_Smartwatch,Clo_Sneakers,2
Clo_Jacket,Hom_Rug,2
Ele_Smartwatch,Hom_Rug,1
Ele_Smartphone,Ele_Smartwatch,1
Hom_Lamp,Hom_Rug,1
Ele_Smartwatch,Clo_Jacket,1
Clo_Jacket,Hom_Coffee Maker,1
Hom_Rug,Boo_Fiction Novel,1
Ele_Smartphone,Clo_Sneakers,1


Part 4: Databricks Interactive Reporting UI

In [0]:
# Create interactive widgets at the top of the notebook
dbutils.widgets.dropdown("Report Type", "monthly", ["daily", "weekly", "monthly"])
dbutils.widgets.text("Start Date (YYYY-MM-DD)", "2024-06-01")
dbutils.widgets.text("End Date (YYYY-MM-DD)", "2024-12-31")

# Extract inputs from dashboard panel
report_type = dbutils.widgets.get("Report Type")
start_date = dbutils.widgets.get("Start Date (YYYY-MM-DD)")
end_date = dbutils.widgets.get("End Date (YYYY-MM-DD)")

def generate_dynamic_summary(report_type, start_dt, end_dt):
    # Constructing time delta differences for historical tracking
    s_dt = pd.to_datetime(start_dt)
    e_dt = pd.to_datetime(end_dt)
    days_diff = (e_dt - s_dt).days
    
    prev_s_dt = (s_dt - pd.Timedelta(days=days_diff)).strftime('%Y-%m-%d')
    prev_e_dt = (s_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    
    # 1. Total Metrics (Current Period)
    curr_metrics = spark.sql(f"""
        SELECT 
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100)), 2) AS total_rev,
            COUNT(DISTINCT o.customer_id) AS unique_cust
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE TO_DATE(o.order_date) BETWEEN '{start_dt}' AND '{end_dt}'
    """).collect()[0]

    # 2. Total Metrics (Previous Period)
    prev_metrics = spark.sql(f"""
        SELECT 
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100)), 2) AS total_rev
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE TO_DATE(o.order_date) BETWEEN '{prev_s_dt}' AND '{prev_e_dt}'
    """).collect()[0]
    
    # 3. Top Products
    top_prods = spark.sql(f"""
        SELECT p.product_name, SUM(oi.quantity) as qty
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN orders o ON oi.order_id = o.order_id
        WHERE TO_DATE(o.order_date) BETWEEN '{start_dt}' AND '{end_dt}'
        GROUP BY p.product_name ORDER BY qty DESC LIMIT 3
    """).toPandas()

    # Percent Differences logic calculation
    rev_change = 0.0
    if prev_metrics['total_rev'] and prev_metrics['total_rev'] > 0:
        rev_change = ((curr_metrics['total_rev'] - prev_metrics['total_rev']) / prev_metrics['total_rev']) * 100

    print(f"====== BUSINESS REPORT SUMMARY ({report_type.upper()}) ======")
    print(f"Reporting Horizon: {start_dt} to {end_dt}")
    print(f"Total Orders:      {curr_metrics['total_orders']}")
    print(f"Revenue Accrued:   ${curr_metrics['total_rev']}")
    print(f"Unique Customers:  {curr_metrics['unique_cust']}")
    print(f"Revenue vs Prev Period: {rev_change:.2f}% Change")
    print("---------------------------------------")
    print("TOP 3 MOVED PRODUCTS:")
    for idx, row in top_prods.iterrows():
        print(f" - {row['product_name']} (Quantity Shifted: {row['qty']})")

# Run Report Generation script automatically using inputs
generate_dynamic_summary(report_type, start_date, end_date)

====== BUSINESS REPORT SUMMARY (MONTHLY) ======
Reporting Horizon: 2024-06-01 to 2024-12-31
Total Orders:      12
Revenue Accrued:   $8865.17
Unique Customers:  9
Revenue vs Prev Period: 0.00% Change
---------------------------------------
TOP 3 MOVED PRODUCTS:
 - Ele_Laptop (Quantity Shifted: 7)
 - Boo_Biography (Quantity Shifted: 6)
 - Boo_Fiction Novel (Quantity Shifted: 6)


Part 5: Edge Case Handling

In [0]:
def run_system_constraints_check():
    print("Executing system constraints sanity tests...")
    
    # Test 1: Identify orphaned order components items referencing dead Orders
    orphaned_count = spark.sql("""
        SELECT COUNT(*) as count FROM order_items WHERE order_id NOT IN (SELECT order_id FROM orders)
    """).collect()[0]['count']
    assert orphaned_count == 0, f"Referential Breach! Found {orphaned_count} structural orphan lines."
    print("✓ Test 1 Passed: Zero orphaned records found.")

    # Test 2: Catch erroneous illegal promotional pricing exceptions (> 100% discount)
    invalid_discount = spark.sql("""
        SELECT COUNT(*) as count FROM order_items WHERE discount_percent > 100 OR discount_percent < 0
    """).collect()[0]['count']
    assert invalid_discount == 0, f"Data Entry Error! Detected {invalid_discount} pricing lines with impossible discount values."
    print("✓ Test 2 Passed: Discount boundaries intact.")

    # Test 3: Check for ghost items (Quantity equals absolute zero)
    zero_qty_count = spark.sql("""
        SELECT COUNT(*) as count FROM order_items WHERE quantity = 0
    """).collect()[0]['count']
    assert zero_qty_count == 0, f"Data Quality Exception: found {zero_qty_count} line items tracking zero asset velocity metrics."
    print("✓ Test 3 Passed: No zero-quantity items found.")

    # Test 4: Time Travel validation checking (Dates skewed towards tomorrow/future dates)
    future_dates = spark.sql("""
        SELECT COUNT(*) as count FROM orders WHERE order_date > CURRENT_TIMESTAMP()
    """).collect()[0]['count']
    assert future_dates == 0, f"Temporal Anomaly! Found {future_dates} transaction records indicating post-dated booking."
    print("✓ Test 4 Passed: No future-dated transactional activity caught.")
    
    print("\n[SUCCESS] The system has passed all business rule verification checks!")

# Execute the test framework block
run_system_constraints_check()

Executing system constraints sanity tests...
✓ Test 1 Passed: Zero orphaned records found.
✓ Test 2 Passed: Discount boundaries intact.
✓ Test 3 Passed: No zero-quantity items found.
✓ Test 4 Passed: No future-dated transactional activity caught.

[SUCCESS] The system has passed all business rule verification checks!
